In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
df = pd.read_csv('TCS_New.csv')

df = df.iloc[2:].reset_index(drop=True)
df.columns = ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']

df['Date'] = pd.to_datetime(df['Date'])
df['Close'] = df['Close'].astype(float)

df = df.sort_values('Date')

In [ ]:
# Returns
df['Returns'] = df['Close'].pct_change()

# Moving averages
df['MA10'] = df['Close'].rolling(10).mean()
df['MA20'] = df['Close'].rolling(20).mean()

# EMA
df['EMA10'] = df['Close'].ewm(span=10).mean()
df['EMA20'] = df['Close'].ewm(span=20).mean()

# Momentum
df['Momentum'] = df['Close'] - df['Close'].shift(10)

# Volatility
df['Volatility'] = df['Returns'].rolling(10).std()

In [ ]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

df['RSI'] = compute_rsi(df['Close'])

In [ ]:
ema12 = df['Close'].ewm(span=12).mean()
ema26 = df['Close'].ewm(span=26).mean()

df['MACD'] = ema12 - ema26
df['MACD_signal'] = df['MACD'].ewm(span=9).mean()

In [ ]:
df['Direction'] = (df['Close'].shift(-5) > df['Close']).astype(int)

df = df.dropna()

In [ ]:
features = df[
    ['Returns','MA10','MA20','EMA10','EMA20',
     'Momentum','Volatility','RSI','MACD','MACD_signal']
].values

target = df['Direction'].values

In [ ]:
split = int(0.8 * len(features))

train_data = features[:split]
test_data = features[split:]

y_train_raw = target[:split]
y_test_raw = target[split:]

In [ ]:
scaler = StandardScaler()

train_data = scaler.fit_transform(train_data)
test_data = scaler.transform(test_data)

In [ ]:
def create_dataset(data, target, time_step=90):
    X, y = [], []
    for i in range(time_step, len(data)):
        X.append(data[i-time_step:i])
        y.append(target[i])
    return np.array(X), np.array(y)

X_train, y_train = create_dataset(train_data, y_train_raw)
X_test, y_test = create_dataset(test_data, y_test_raw)

In [ ]:
print(X_train.shape)  # (samples, 90, features)

(883, 90, 10)


In [ ]:
model = Sequential()

model.add(Bidirectional(LSTM(256, return_sequences=True),
                        input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.4))

model.add(Bidirectional(LSTM(128)))
model.add(Dropout(0.4))

model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
early_stop = EarlyStopping(patience=7, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(patience=3, factor=0.5)

history = model.fit(
    X_train, y_train,
    epochs=80,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, lr_scheduler]
)

Epoch 1/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 7s 46ms/step - accuracy: 0.5696 - loss: 0.6816 - val_accuracy: 0.5065 - val_loss: 0.7292 - learning_rate: 0.0010
Epoch 2/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.6444 - loss: 0.6462 - val_accuracy: 0.3312 - val_loss: 0.9223 - learning_rate: 0.0010
Epoch 3/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.6399 - loss: 0.6308 - val_accuracy: 0.4286 - val_loss: 0.8914 - learning_rate: 0.0010
Epoch 4/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.6795 - loss: 0.6013 - val_accuracy: 0.4805 - val_loss: 0.9389 - learning_rate: 0.0010
Epoch 5/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.7259 - loss: 0.5548 - val_accuracy: 0.4156 - val_loss: 0.9983 - learning_rate: 5.0000e-04
Epoch 6/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.7554 - loss: 0.5121 - val_accuracy: 0.3961 - val_loss: 1.1545 - learning_rate: 5.0000e-04
Epoch 7/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.7622 - loss: 0.4963 -

In [ ]:
probs = model.predict(X_test).flatten()

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 134ms/step


In [ ]:
best_acc = 0
best_t = 0.5

for t in np.arange(0.3, 0.7, 0.01):
    preds = (probs > t).astype(int)
    acc = accuracy_score(y_test, preds)

    if acc > best_acc:
        best_acc = acc
        best_t = t

print("Best Threshold:", best_t)
print("Best Accuracy:", best_acc)

Best Threshold: 0.6600000000000004
Best Accuracy: 0.564935064935065


In [ ]:
preds = (probs > best_t).astype(int)

print("Accuracy:", accuracy_score(y_test, preds))
print("Precision:", precision_score(y_test, preds))
print("Recall:", recall_score(y_test, preds))

Accuracy: 0.564935064935065
Precision: 0.5
Recall: 0.07462686567164178


In [ ]:
upper = np.percentile(probs, 80)
lower = np.percentile(probs, 20)

mask = (probs >= upper) | (probs <= lower)

filtered_preds = preds[mask]
filtered_actual = y_test[mask]

print("Filtered Accuracy:", accuracy_score(filtered_actual, filtered_preds))
print("Trades:", len(filtered_preds))

Filtered Accuracy: 0.6774193548387096
Trades: 62


In [ ]:
!pip install xgboost

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score

In [ ]:
from tensorflow.keras.layers import Input

inputs = Input(shape=(X_train.shape[1], X_train.shape[2]))

x = Bidirectional(LSTM(256, return_sequences=True))(inputs)
x = Dropout(0.4)(x)

x = Bidirectional(LSTM(128))(x)
x = Dropout(0.4)(x)

x = Dense(64, activation='relu')(x)

outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)

In [ ]:
feature_model = Model(inputs=model.input, outputs=x)

In [ ]:
X_train_lstm = feature_model.predict(X_train)
X_test_lstm = feature_model.predict(X_test)

28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step


In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8
)

xgb_model.fit(X_train_lstm, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
xgb_preds = xgb_model.predict(X_test_lstm)

In [ ]:
print("XGBoost Accuracy:", accuracy_score(y_test, xgb_preds))

XGBoost Accuracy: 0.4155844155844156


In [ ]:
lstm_probs = model.predict(X_test).flatten()

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 89ms/step


In [ ]:
xgb_probs = xgb_model.predict_proba(X_test_lstm)[:,1]

In [ ]:
final_probs = 0.5 * lstm_probs + 0.5 * xgb_probs

In [ ]:
final_probs = 0.5 * lstm_probs + 0.5 * xgb_probs

In [ ]:
best_acc = 0
best_t = 0.5

for t in np.arange(0.3, 0.7, 0.01):
    preds = (final_probs > t).astype(int)
    acc = accuracy_score(y_test, preds)

    if acc > best_acc:
        best_acc = acc
        best_t = t

print("Best Threshold:", best_t)
print("Best Accuracy:", best_acc)

Best Threshold: 0.6500000000000004
Best Accuracy: 0.487012987012987


In [ ]:
confidence = np.abs(final_probs - best_t)

mask = confidence > 0.03

filtered_preds = preds[mask]
filtered_actual = y_test[mask]

print("Filtered Accuracy:", accuracy_score(filtered_actual, filtered_preds))
print("Trades:", len(filtered_preds))

Filtered Accuracy: 0.4496124031007752
Trades: 129


In [ ]:
# Flatten sequence data
X_train_xgb = X_train.reshape(X_train.shape[0], -1)
X_test_xgb = X_test.reshape(X_test.shape[0], -1)

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05
)

xgb_model.fit(X_train_xgb, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
# XGBoost predictions
xgb_preds = xgb_model.predict(X_test_xgb)

# Probabilities (important for advanced metrics)
xgb_probs = xgb_model.predict_proba(X_test_xgb)[:, 1]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(y_test, xgb_preds))
print("Precision:", precision_score(y_test, xgb_preds))
print("Recall:", recall_score(y_test, xgb_preds))
print("F1 Score:", f1_score(y_test, xgb_preds))

Accuracy: 0.4090909090909091
Precision: 0.36666666666666664
Recall: 0.4925373134328358
F1 Score: 0.42038216560509556


In [ ]:
best_acc = 0
best_t = 0.5

for t in np.arange(0.3, 0.7, 0.01):
    preds = (xgb_probs > t).astype(int)
    acc = accuracy_score(y_test, preds)

    if acc > best_acc:
        best_acc = acc
        best_t = t

print("Best Threshold:", best_t)
print("Best Accuracy:", best_acc)

Best Threshold: 0.3
Best Accuracy: 0.4675324675324675


In [ ]:
confidence = np.abs(xgb_probs - best_t)

mask = confidence > 0.05   # tune this

filtered_preds = (xgb_probs > best_t).astype(int)[mask]
filtered_actual = y_test[mask]

if len(filtered_preds) > 0:
    print("Filtered Accuracy:", accuracy_score(filtered_actual, filtered_preds))
    print("Trades:", len(filtered_preds))
else:
    print("No trades selected")

Filtered Accuracy: 0.4520547945205479
Trades: 146


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, xgb_preds))

              precision    recall  f1-score   support

           0       0.47      0.34      0.40        87
           1       0.37      0.49      0.42        67

    accuracy                           0.41       154
   macro avg       0.42      0.42      0.41       154
weighted avg       0.42      0.41      0.41       154

